# ReServe AWQ / FP16 timing profiling

This replaces the NF4 timing notebook with **AWQ W4A16** and a matching FP16 control. Use one Kaggle Tesla T4 (GPU 0); enable Internet. Qwen publishes the prepared AWQ checkpoint, so this notebook does not run quantization calibration.

**Run order:** install once → restart the Python kernel → run Steps 2–5 with `MODE = "awq"` → download the ZIP → restart the kernel → run Steps 2–5 with `MODE = "fp16"` → download the second ZIP. If you start a new Kaggle session, install again. Installation changes this notebook environment's packages.

The batch sizes, context lengths, shuffled order, token ID, 10 warm-ups, and 50 measured trials match your original sweep. Decode measures one token using a fresh cache of exactly the requested length. Prefill/cache construction and token selection stay outside the decode timer.

**Rerun both modes:** the AWQ-compatible stack differs from your old Torch 2.11 / Transformers 5.20 stack. Keep old results as historical records; use new FP16 and AWQ profiles together for calibration. These are fixed-configuration profiles, not a live precision-switch benchmark.

## 1. Install once, then restart the kernel

AutoAWQ's last documented tested versions are Torch 2.6.0 and Transformers 4.51.3. We pin that stack, use CUDA 12.4 wheels, and select its Triton GEMM backend explicitly. No compiled `autoawq-kernels` extension or FlashAttention installation is required. The first installation can download large PyTorch packages.

After this cell succeeds, **restart the Python kernel** and continue at Step 2. Do not run the old NF4 dependency cell afterward.

In [ ]:
import subprocess
import sys

if not ((3, 10) <= sys.version_info[:2] <= (3, 13)):
    raise RuntimeError("Use a Kaggle Python 3.10-3.13 kernel for this pinned profile environment")

# Install the documented AutoAWQ-compatible torch stack, including matching
# vision/audio packages so stale Kaggle extensions do not break model imports.
subprocess.check_call([
    sys.executable, "-m", "pip", "install", "--upgrade",
    "torch==2.6.0", "torchvision==0.21.0", "torchaudio==2.6.0",
    "--index-url", "https://download.pytorch.org/whl/cu124",
])
subprocess.check_call([
    sys.executable, "-m", "pip", "install", "--upgrade",
    "torch==2.6.0", "triton==3.2.0", "autoawq==0.2.9",
    "transformers==4.51.3", "accelerate==1.6.0", "huggingface-hub==0.30.2",
    "tokenizers==0.21.1", "safetensors==0.5.3", "numpy==2.2.6", "pandas==2.2.3",
    "datasets==3.5.0", "librosa==0.11.0",
])
print("Installation finished. RESTART THE PYTHON KERNEL now.")
print("After restarting, start at Step 2; do not rerun this installation cell.")

## 2. Select the precision

Start with `MODE = "awq"`. After downloading that run and restarting the kernel, change it to `"fp16"`. Keep all other settings identical. `SMOKE_TEST_ONLY = True` runs loading and a small prefill/decode compatibility check without the full sweep; the default `False` performs that check and then profiles.

In [ ]:
MODE = "awq"                  # "awq" first; restart, then "fp16"
EXPECTED_GPU = "Tesla T4"
SMOKE_TEST_ONLY = False
WARMUP_RUNS = 10
MEASURED_RUNS = 50
CASE_ORDER = [
    [2, 2048], [4, 512], [2, 512], [2, 1024],
    [8, 1024], [4, 1024], [8, 256], [4, 256],
    [1, 512], [1, 1024], [8, 512], [8, 2048],
    [2, 256], [4, 2048], [1, 256], [1, 2048],
]

## 3. Define the profiler

This cell loads no model. It defines the pinned loader, actual packed-layer checks, FP16 KV checks, storage inventory, fresh-cache timing loop, and incremental result saving. It accepts both older and newer Transformers KV-cache layouts.

AWQ model-level attention/MLP fusion is disabled, preserving ordinary decoder-layer boundaries and inspectable KV. AutoAWQ's internal GEMM heuristic can dequantize a projection temporarily for large token batches; timing and workspace measurements include that behavior. The low-precision weights remain packed at rest.

In [ ]:
"""Shared, embedded helpers for ReServe's fixed FP16/AWQ notebook profiles."""
import gc
import itertools
import json
import os
import platform
import random
import re
import statistics
import time
from datetime import datetime, timezone
from importlib.metadata import version, PackageNotFoundError
from pathlib import Path

import pandas as pd
import torch
import transformers
from transformers import AutoConfig, AutoModelForCausalLM, AutoTokenizer, AwqConfig

MODEL_IDS = {
    "fp16": "Qwen/Qwen2.5-1.5B-Instruct",
    "awq": "Qwen/Qwen2.5-1.5B-Instruct-AWQ",
}
MODEL_REVISIONS = {
    "fp16": "989aa7980e4cf806f80c7fef2b1adb7bc71aa306",
    "awq": "c5b72040769bfde6c5b562ae890f9657b0803f79",
}
PACKAGE_PINS = {
    "torch": "2.6.0", "transformers": "4.51.3", "autoawq": "0.2.9",
    "triton": "3.2.0", "accelerate": "1.6.0", "huggingface-hub": "0.30.2",
    "numpy": "2.2.6", "pandas": "2.2.3",
}
INPUT_TOKEN_ID = 14990
EXPECTED_KV_BYTES_PER_TOKEN = 28672


def prepare_environment(expected_gpu="Tesla T4"):
    actual_packages = {}
    for name, expected in PACKAGE_PINS.items():
        try:
            actual_packages[name] = version(name)
        except PackageNotFoundError:
            actual_packages[name] = "missing"
    wrong = {name: (actual_packages[name], expected) for name, expected in PACKAGE_PINS.items()
             if actual_packages[name].split("+")[0] != expected}
    if wrong or transformers.__version__ != PACKAGE_PINS["transformers"] or torch.__version__.split("+")[0] != PACKAGE_PINS["torch"]:
        raise RuntimeError(
            f"Pinned AWQ environment is not active: {wrong}. Run the installation cell, "
            "restart the Python kernel, then run from the settings cell."
        )
    if not torch.cuda.is_available():
        raise RuntimeError("Enable a Kaggle NVIDIA GPU. This profiler requires CUDA.")
    device = torch.device("cuda:0")
    torch.cuda.set_device(device)
    gpu = torch.cuda.get_device_name(device)
    if expected_gpu is not None and gpu != expected_gpu:
        raise RuntimeError(f"Expected {expected_gpu}, got {gpu}. Use the same GPU for both modes.")
    if tuple(torch.cuda.get_device_capability(device)) < (7, 5):
        raise RuntimeError("This AWQ configuration targets T4-class compute capability 7.5 or newer.")
    if torch.version.cuda != "12.4":
        raise RuntimeError(f"Expected the cu124 wheel, got CUDA runtime {torch.version.cuda}.")
    torch.manual_seed(42)
    random.seed(42)
    torch.backends.cuda.matmul.allow_tf32 = False
    torch.backends.cudnn.allow_tf32 = False
    torch.cuda.synchronize(device)
    gc.collect()
    torch.cuda.empty_cache()
    if torch.cuda.memory_allocated(device) != 0:
        raise RuntimeError("GPU tensors already exist. Restart the kernel before profiling another mode.")
    return device, {
        "gpu": gpu, "compute_capability": list(torch.cuda.get_device_capability(device)),
        "torch": torch.__version__, "cuda_runtime": torch.version.cuda,
        "transformers": transformers.__version__, "python": platform.python_version(),
        "packages": actual_packages,
    }


def cache_pairs(cache):
    """Read both older key_cache/value_cache and newer .layers cache layouts."""
    if cache is None:
        raise ValueError("Model did not return a KV cache")
    if hasattr(cache, "layers"):
        return [(layer.keys, layer.values) for layer in cache.layers]
    if hasattr(cache, "key_cache") and hasattr(cache, "value_cache"):
        if len(cache.key_cache) != len(cache.value_cache):
            raise ValueError("Unequal key/value cache layer counts")
        return list(zip(cache.key_cache, cache.value_cache))
    if isinstance(cache, (tuple, list)):
        if any(len(layer) != 2 for layer in cache):
            raise ValueError("Expected exactly keys and values in each legacy cache layer")
        return [(layer[0], layer[1]) for layer in cache]
    raise ValueError(f"Unsupported cache type: {type(cache).__name__}")


def inspect_cache(cache, batch_size, length, config):
    pairs = cache_pairs(cache)
    if len(pairs) != config.num_hidden_layers:
        raise ValueError(f"Expected {config.num_hidden_layers} KV layers, got {len(pairs)}")
    head_dim = getattr(config, "head_dim", None) or config.hidden_size // config.num_attention_heads
    expected_shape = (batch_size, config.num_key_value_heads, length, head_dim)
    logical_bytes, unique_storages, details = 0, {}, []
    for index, pair in enumerate(pairs):
        for kind, tensor in zip(("keys", "values"), pair):
            if tuple(tensor.shape) != expected_shape:
                raise ValueError(f"Layer {index} {kind}: {tuple(tensor.shape)} != {expected_shape}")
            if tensor.dtype != torch.float16 or str(tensor.device) != "cuda:0":
                raise ValueError(f"Expected FP16 KV on cuda:0; got {tensor.dtype}, {tensor.device}")
            logical = int(tensor.numel() * tensor.element_size())
            storage = tensor.untyped_storage()
            key = (str(tensor.device), int(storage.data_ptr()))
            backing = int(storage.nbytes())
            seen = key in unique_storages
            unique_storages[key] = backing
            logical_bytes += logical
            details.append({
                "layer": index, "kind": kind, "shape": json.dumps(list(tensor.shape)),
                "dtype": str(tensor.dtype), "logical_bytes": logical,
                "backing_storage_bytes": backing, "storage_seen_previously": seen,
            })
    expected = 2 * config.num_hidden_layers * config.num_key_value_heads * head_dim * 2 * batch_size * length
    if logical_bytes != expected:
        raise ValueError(f"KV size {logical_bytes} != architecture-derived {expected}")
    return {
        "kv_logical_bytes": logical_bytes, "kv_unique_backing_bytes": sum(unique_storages.values()),
        "kv_unique_storages": len(unique_storages),
        "kv_bytes_per_cached_token": logical_bytes // (batch_size * length),
        "kv_dtype": "float16", "cached_tokens_per_request": length,
    }, details


def model_inventory(model):
    """Include packed AWQ buffers; count tied/shared backing storage once."""
    seen, rows, layers = set(), [], {}
    for role, iterator in (
        ("parameter", model.named_parameters(remove_duplicate=False)),
        ("buffer", model.named_buffers(remove_duplicate=False)),
    ):
        for name, tensor in iterator:
            match = re.match(r"^model\.layers\.(\d+)\.", name)
            layer = int(match.group(1)) if match else -1
            storage = tensor.untyped_storage()
            key = (str(tensor.device), int(storage.data_ptr()), int(storage.nbytes()))
            unique = key not in seen
            seen.add(key)
            charged = int(storage.nbytes()) if unique else 0
            rows.append({
                "tensor_name": name, "role": role, "decoder_layer": layer,
                "dtype": str(tensor.dtype), "shape": json.dumps(list(tensor.shape)),
                "numel": int(tensor.numel()), "logical_bytes": int(tensor.numel() * tensor.element_size()),
                "backing_storage_bytes": int(storage.nbytes()), "unique_backing_bytes": charged,
                "shared_storage": not unique,
            })
            item = layers.setdefault(layer, {"decoder_layer": layer, "unique_backing_bytes": 0,
                                             "parameter_backing_bytes": 0, "buffer_backing_bytes": 0,
                                             "awq_linear_count": 0})
            item["unique_backing_bytes"] += charged
            item[role + "_backing_bytes"] += charged
    for name, module in model.named_modules():
        if all(hasattr(module, attr) for attr in ("qweight", "qzeros", "scales")):
            match = re.match(r"^model\.layers\.(\d+)\.", name)
            layer = int(match.group(1)) if match else -1
            layers[layer]["awq_linear_count"] += 1
    return rows, [layers[key] for key in sorted(layers)]


def load_profile_model(mode, revision=None):
    if mode not in MODEL_IDS:
        raise ValueError('MODE must be "fp16" or "awq"')
    model_id = MODEL_IDS[mode]
    revision = revision or MODEL_REVISIONS[mode]
    config = AutoConfig.from_pretrained(model_id, revision=revision, trust_remote_code=False)
    resolved = getattr(config, "_commit_hash", None)
    if resolved != revision:
        raise ValueError(f"Requested commit {revision}, config resolved to {resolved}")
    fields = ("model_type", "hidden_size", "intermediate_size", "num_hidden_layers",
              "num_attention_heads", "num_key_value_heads", "vocab_size", "tie_word_embeddings",
              "max_position_embeddings", "rope_theta", "rms_norm_eps")
    if mode == "awq":
        base_config = AutoConfig.from_pretrained(MODEL_IDS["fp16"], revision=MODEL_REVISIONS["fp16"], trust_remote_code=False)
        mismatches = {key: (getattr(config, key, None), getattr(base_config, key, None))
                      for key in fields if getattr(config, key, None) != getattr(base_config, key, None)}
        if mismatches:
            raise ValueError(f"FP16/AWQ architecture mismatch: {mismatches}")
        quant = getattr(config, "quantization_config", {})
        if hasattr(quant, "to_dict"):
            quant = quant.to_dict()
        for key, expected in {"quant_method": "awq", "bits": 4, "group_size": 128,
                              "zero_point": True, "version": "gemm"}.items():
            if quant.get(key) != expected:
                raise ValueError(f"AWQ checkpoint {key}={quant.get(key)!r}; expected {expected!r}")
        from awq.modules.linear import gemm as awq_gemm
        # Deliberately choose Triton: no torch-ABI-specific autoawq-kernels wheel.
        awq_gemm.awq_ext = None
        if not awq_gemm.TRITON_AVAILABLE:
            raise RuntimeError("AWQ Triton kernels unavailable; refusing a naive slow fallback")
        options = {"quantization_config": AwqConfig(
            bits=4, group_size=128, zero_point=True, version="gemm", backend="autoawq",
            do_fuse=False, modules_to_not_convert=["lm_head"],
        )}
    else:
        options = {}
        if getattr(config, "quantization_config", None):
            raise ValueError("FP16 source unexpectedly contains quantization metadata")
    model = AutoModelForCausalLM.from_pretrained(
        model_id, revision=revision, config=config, torch_dtype=torch.float16,
        device_map={"": 0}, attn_implementation="sdpa", low_cpu_mem_usage=True,
        trust_remote_code=False, use_safetensors=True, **options,
    ).eval()
    if any(str(t.device) != "cuda:0" for t in itertools.chain(model.parameters(), model.buffers())):
        raise ValueError("Every parameter and buffer must reside on GPU 0; no CPU/disk offload")
    if any(p.dtype != torch.float16 for p in model.parameters() if p.is_floating_point()):
        raise ValueError("Floating parameters must use FP16")
    quantized = [(name, module) for name, module in model.named_modules()
                 if all(hasattr(module, attr) for attr in ("qweight", "qzeros", "scales"))]
    if mode == "awq":
        from awq.modules.linear.gemm import WQLinear_GEMM
        if len(quantized) != 196 or any(not isinstance(m, WQLinear_GEMM) for _, m in quantized):
            raise ValueError(f"Expected 196 unfused AWQ GEMM linear modules, got {len(quantized)}")
        for name, module in quantized:
            if module.w_bit != 4 or module.group_size != 128:
                raise ValueError(f"Wrong AWQ format in {name}")
            if module.qweight.dtype != torch.int32 or module.qzeros.dtype != torch.int32 or module.scales.dtype != torch.float16:
                raise ValueError(f"Wrong packed-weight/metadata dtypes in {name}")
        if getattr(model, "_awq_is_fused", False):
            raise ValueError("Attention/MLP fusion must remain disabled for cache and layer-level inspection")
    elif quantized or getattr(model, "is_quantized", False):
        raise ValueError("FP16 mode unexpectedly contains quantized modules")
    tokenizer = AutoTokenizer.from_pretrained(model_id, revision=revision, trust_remote_code=False)
    if tokenizer.encode("hello", add_special_tokens=False) != [INPUT_TOKEN_ID]:
        raise ValueError("Tokenizer differs from the original profiling input")
    inventory, layer_bytes = model_inventory(model)
    if mode == "awq" and any(row["awq_linear_count"] != 7 for row in layer_bytes if row["decoder_layer"] >= 0):
        raise ValueError("Each of the 28 decoder layers must contain seven AWQ projections")
    quant = getattr(model.config, "quantization_config", None)
    identity = {
        "model_id": model_id, "model_revision": revision, "resolved_config_revision": resolved,
        "fp16_reference_model_id": MODEL_IDS["fp16"], "fp16_reference_revision": MODEL_REVISIONS["fp16"],
        "quantization_config": quant.to_dict() if hasattr(quant, "to_dict") else quant,
        "quantized_linear_layers": len(quantized), "model_config": model.config.to_dict(),
        "awq_kernel_backend": "autoawq_0.2.9_gemm_triton" if mode == "awq" else None,
        "awq_fusion_enabled": False, "attention_implementation_requested": "sdpa",
        "attention_implementation_actual": model.config._attn_implementation,
        "resident_tensor_unique_backing_bytes": sum(row["unique_backing_bytes"] for row in inventory),
        "comparison_note": "Rerun FP16 on this same pinned stack. Earlier 2.11/5.20 profiles are historical only.",
        "checkpoint_lineage_note": "Official FP16 and official prequantized AWQ are separately pinned repositories; identical quantization-source FP16 commit is not independently established.",
        "awq_execution_note": "AutoAWQ GEMM uses Triton packed GEMM for small token batches and dequantize+FP16 matmul for batch*sequence >=1024. Workspace/timing include this backend heuristic.",
        "layer_variant_note": "Inventory includes all layer parameters and buffers, including FP16 tensors whose values may have been rescaled during AWQ preparation.",
    }
    return model, identity, inventory, layer_bytes


def write_inventory(directory, inventory, layer_bytes):
    pd.DataFrame(inventory).to_csv(directory / "model_tensors.csv", index=False)
    pd.DataFrame(layer_bytes).to_csv(directory / "layer_memory.csv", index=False)


@torch.inference_mode()
def smoke_check(model, device):
    ids = mask = output = cache = token = decoded = decode_mask = None
    try:
        ids = torch.full((1, 16), INPUT_TOKEN_ID, dtype=torch.long, device=device)
        mask = torch.ones_like(ids)
        output = model(input_ids=ids, attention_mask=mask, use_cache=True, logits_to_keep=1)
        cache = output.past_key_values
        first, _ = inspect_cache(cache, 1, 16, model.config)
        token = output.logits[:, -1, :].argmax(dim=-1, keepdim=True)
        if not torch.isfinite(output.logits).all().item():
            raise ValueError("Non-finite prefill logits")
        output = None
        decode_mask = torch.ones((1, 17), dtype=torch.long, device=device)
        decoded = model(input_ids=token, attention_mask=decode_mask, past_key_values=cache,
                        use_cache=True, logits_to_keep=1)
        second, _ = inspect_cache(decoded.past_key_values, 1, 17, model.config)
        if not torch.isfinite(decoded.logits).all().item():
            raise ValueError("Non-finite decode logits")
        if first["kv_bytes_per_cached_token"] != EXPECTED_KV_BYTES_PER_TOKEN:
            raise ValueError("Unexpected KV bytes per cached token")
        if second["kv_logical_bytes"] - first["kv_logical_bytes"] != EXPECTED_KV_BYTES_PER_TOKEN:
            raise ValueError("Incorrect one-token cache growth")
        return {"kv_dtype_verified": "float16", "kv_bytes_per_token_verified": EXPECTED_KV_BYTES_PER_TOKEN,
                "cache_type": type(cache).__name__, "smoke_test": "passed"}
    finally:
        ids = mask = output = cache = token = decoded = decode_mask = None
        gc.collect()
        torch.cuda.empty_cache()
        torch.cuda.synchronize(device)


def summarize(samples):
    names = ["median_ms", "mean_ms", "std_ms", "q1_ms", "q3_ms", "iqr_ms",
             "min_ms", "max_ms", "cv_pct", "peak_allocated_mib", "peak_reserved_mib"]
    result = {name: None for name in names}
    result["measured_runs"] = len(samples)
    if not samples:
        return result
    values = [sample["latency_ms"] for sample in samples]
    q1, _, q3 = statistics.quantiles(values, n=4, method="inclusive") if len(values) > 1 else [values[0]] * 3
    mean = statistics.mean(values)
    std = statistics.stdev(values) if len(values) > 1 else None
    result.update({
        "median_ms": statistics.median(values), "mean_ms": mean, "std_ms": std,
        "q1_ms": q1, "q3_ms": q3, "iqr_ms": q3 - q1,
        "min_ms": min(values), "max_ms": max(values),
        "cv_pct": 100 * std / mean if std is not None and mean else None,
        "peak_allocated_mib": max(s["peak_allocated_mib"] for s in samples),
        "peak_reserved_mib": max(s["peak_reserved_mib"] for s in samples),
    })
    return result


@torch.inference_mode()
def measure_once(model, phase, ids, prompt_mask, decode_mask, device):
    prefix = cache = next_token = output = None
    try:
        if phase == "decode":
            # A NEW cache for every trial: context length never accumulates.
            # Both prefill and token selection are outside the decode timer.
            prefix = model(input_ids=ids, attention_mask=prompt_mask,
                           use_cache=True, logits_to_keep=1)
            cache = prefix.past_key_values
            assert cache is not None
            next_token = prefix.logits[:, -1, :].argmax(dim=-1, keepdim=True)
            assert tuple(next_token.shape) == (ids.shape[0], 1)
            prefix = None

        torch.cuda.synchronize(device)
        torch.cuda.reset_peak_memory_stats(device)
        start = time.perf_counter()
        if phase == "prefill":
            output = model(input_ids=ids, attention_mask=prompt_mask,
                           use_cache=True, logits_to_keep=1)
        else:
            output = model(input_ids=next_token, attention_mask=decode_mask,
                           past_key_values=cache, use_cache=True, logits_to_keep=1)
        torch.cuda.synchronize(device)
        elapsed_ms = (time.perf_counter() - start) * 1000
        assert output.logits.shape[0] == ids.shape[0]
        return {
            "latency_ms": elapsed_ms,
            "peak_allocated_mib": torch.cuda.max_memory_allocated(device) / 1024**2,
            "peak_reserved_mib": torch.cuda.max_memory_reserved(device) / 1024**2,
        }
    finally:
        prefix = cache = next_token = output = None


def checkpoint(directory, summaries, raw_rows, metadata):
    raw_columns = ["run_id", "mode", "phase", "batch_size", "length_tokens", "trial",
                   "latency_ms", "peak_allocated_mib", "peak_reserved_mib"]
    outputs = {
        "summary.csv": pd.DataFrame(summaries).sort_values(["phase", "batch_size", "length_tokens"]),
        "raw_timings.csv": pd.DataFrame(raw_rows, columns=raw_columns),
    }
    for name, frame in outputs.items():
        temporary = directory / (name + ".tmp")
        frame.to_csv(temporary, index=False)
        temporary.replace(directory / name)
    temporary = directory / "metadata.json.tmp"
    temporary.write_text(json.dumps(metadata, indent=2), encoding="utf-8")
    temporary.replace(directory / "metadata.json")


def sweep(model, directory, metadata, cases, token_id, device, warmups, repeats):
    summaries, raw_rows = [], []
    for batch_size, length in cases:
        for phase in ("prefill", "decode"):
            case = {"run_id": metadata["run_id"], "mode": metadata["mode"], "phase": phase,
                    "batch_size": batch_size, "length_tokens": length}
            samples, status, error = [], "ok", ""
            completed_warmups = 0
            fatal_error = None
            ids = prompt_mask = decode_mask = None
            print(f"Running {phase}: batch={batch_size}, length={length}", flush=True)
            try:
                ids = torch.full((batch_size, length), token_id, dtype=torch.long, device=device)
                prompt_mask = torch.ones_like(ids)
                decode_mask = torch.ones((batch_size, length + 1), dtype=torch.long, device=device)
                for _ in range(warmups):
                    measure_once(model, phase, ids, prompt_mask, decode_mask, device)
                    completed_warmups += 1
                for trial in range(repeats):
                    sample = measure_once(model, phase, ids, prompt_mask, decode_mask, device)
                    samples.append(sample)
                    raw_rows.append({**case, "trial": trial + 1, **sample})
            except torch.cuda.OutOfMemoryError as exc:
                status, error = "oom", str(exc).splitlines()[0]
            except KeyboardInterrupt as exc:
                status, error, fatal_error = "interrupted", "User interrupted this case", exc
            except Exception as exc:
                status, error, fatal_error = "error", f"{type(exc).__name__}: {exc}", exc
            finally:
                ids = prompt_mask = decode_mask = None
                gc.collect()
                torch.cuda.empty_cache()

            row = {**case, "status": status, "error": error,
                   "warmups_completed": completed_warmups, **summarize(samples)}
            summaries.append(row)
            metadata["cases_recorded"] = len(summaries)
            metadata["run_status"] = status if fatal_error is not None else "running"
            checkpoint(directory, summaries, raw_rows, metadata)
            if samples:
                print(f"{status} | n={len(samples)} | median={row['median_ms']:.2f} ms | "
                      f"IQR={row['iqr_ms']:.2f} ms", flush=True)
            else:
                print(f"{status} | {error}", flush=True)
            if fatal_error is not None:
                raise fatal_error

    metadata["run_status"] = "completed_with_oom" if any(r["status"] == "oom" for r in summaries) else "completed"
    metadata["finished_at_utc"] = datetime.now(timezone.utc).isoformat()
    checkpoint(directory, summaries, raw_rows, metadata)
    return pd.DataFrame(summaries).sort_values(["phase", "batch_size", "length_tokens"])





def run_precision_profile(mode, expected_gpu="Tesla T4", smoke_only=False):
    if mode not in MODEL_IDS:
        raise ValueError('MODE must be "fp16" or "awq"')
    if MEASURED_RUNS < 2 or WARMUP_RUNS < 0:
        raise ValueError("Use at least two measurements and nonnegative warm-ups")
    device, environment = prepare_environment(expected_gpu)
    run_id = f"timing_{mode}_" + datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S_%fZ")
    directory = Path("/kaggle/working/reserve_profiles") / run_id
    directory.mkdir(parents=True, exist_ok=False)
    metadata = {
        "schema_version": 2, "run_id": run_id, "mode": mode, "run_status": "loading",
        "model_id": MODEL_IDS[mode], "requested_revision": MODEL_REVISIONS[mode],
        "environment": environment, "torch": torch.__version__,
        "transformers": transformers.__version__, "cuda_runtime": torch.version.cuda,
        "gpu": torch.cuda.get_device_name(device), "python": platform.python_version(),
        "gpu_total_bytes": torch.cuda.get_device_properties(device).total_memory,
        "warmup_runs": WARMUP_RUNS, "measured_runs": MEASURED_RUNS,
        "grid_seed": 42, "case_order": CASE_ORDER, "input_token_id": INPUT_TOKEN_ID,
        "expected_cases": len(CASE_ORDER) * 2, "expected_measured_samples": len(CASE_ORDER) * 2 * MEASURED_RUNS,
        "reference_fp16_run_id": "fp16_20261126T204405_157396Z",
        "historical_results_comparable": False,
        "timing": "synchronized forward-pass wall time including host launch overhead",
        "decode_setup": "fresh prefill cache and argmax prepared outside the decode timer",
        "memory": "per-trial PyTorch process peaks; decode includes its pre-existing cache",
        "coverage": "homogeneous fixed configurations; no live switching, layer sensitivity, or paged KV allocator",
        "allocator_environment": {name: os.environ.get(name) for name in ("PYTORCH_ALLOC_CONF", "PYTORCH_CUDA_ALLOC_CONF", "CUDA_VISIBLE_DEVICES")},
    }
    def save_metadata():
        temporary = directory / "metadata.json.tmp"
        temporary.write_text(json.dumps(metadata, indent=2))
        temporary.replace(directory / "metadata.json")
    save_metadata()
    model = None
    try:
        allocated_before_load = torch.cuda.memory_allocated(device)
        model, identity, inventory, layer_bytes = load_profile_model(mode)
        metadata.update(identity)
        write_inventory(directory, inventory, layer_bytes)
        if max(length for _, length in CASE_ORDER) + 1 > model.config.max_position_embeddings:
            raise ValueError("Grid exceeds model context length")
        metadata.update(smoke_check(model, device))
        metadata["allocated_before_load_mib"] = allocated_before_load / 1024**2
        metadata["baseline_allocated_mib"] = torch.cuda.memory_allocated(device) / 1024**2
        metadata["run_status"] = "smoke_only_completed" if smoke_only else "running"
        save_metadata()
        print(f"Smoke check passed: mode={mode}; KV=FP16, 28 KiB/token; AWQ modules={identity['quantized_linear_layers']}")
        print(f"Saving to {directory}", flush=True)
        if smoke_only:
            return pd.DataFrame(), directory
        frame = sweep(model, directory, metadata, CASE_ORDER, INPUT_TOKEN_ID,
                      device, WARMUP_RUNS, MEASURED_RUNS)
        return frame, directory
    except BaseException as exc:
        metadata["run_status"] = "interrupted" if isinstance(exc, KeyboardInterrupt) else "error"
        metadata["error"] = f"{type(exc).__name__}: {exc}"
        save_metadata()
        print(f"Partial files retained in {directory}", flush=True)
        raise
    finally:
        model = None
        gc.collect()
        torch.cuda.empty_cache()

## 4. Run this mode

The loader verifies 196 real AWQ projections (seven per decoder layer), packed INT32 containers, FP16 scales, FP16 KV, finite logits, and one-token cache growth before entering the sweep. Errors stop with saved diagnostics. OOM cases preserve completed samples and are marked `oom`; calibrate only from complete `status == "ok"` cases.

In [ ]:
profile_summary_df, profile_output_dir = run_precision_profile(
    mode=MODE, expected_gpu=EXPECTED_GPU, smoke_only=SMOKE_TEST_ONLY,
)
display(profile_summary_df.round(3))
print("Results:", profile_output_dir)

## 5. Download the results

Download the ZIP from Kaggle's Output/files panel before restarting for the other mode. Each full run has `summary.csv`, `raw_timings.csv`, `metadata.json`, `model_tensors.csv`, and `layer_memory.csv`. Smoke-only runs contain metadata and inventories but no timing calibration data.

In [ ]:
import shutil
zip_path = shutil.make_archive(str(profile_output_dir), "zip", root_dir=profile_output_dir)
print("Download:", zip_path)

## Reading and comparing results

- `summary.csv`: per-shape prefill/decode statistics, completed sample counts, and PyTorch memory peaks.
- `raw_timings.csv`: up to 1,600 measured forward-pass samples per mode.
- `metadata.json`: exact model repositories/commits, package stack, kernel choice, smoke checks, and run status.
- `model_tensors.csv`: logical tensor size and globally deduplicated backing size, including AWQ `qweight`, `qzeros`, and `scales` buffers.
- `layer_memory.csv`: resident parameter/buffer bytes by decoder layer; layer `-1` contains tensors outside the decoder blocks.

Compare FP16/AWQ only when GPU, software, grid, and profiling settings match. The official AWQ checkpoint and FP16 reference are separately pinned; the publisher's exact original FP16 quantization-source commit is not independently established. Equal architecture does not establish exact checkpoint lineage.

This notebook measures the two endpoints. Intermediate layer configurations, copy/replacement costs, live KV preservation, elastic allocator behavior, and output quality require their own later measurements. Quantized layers may also contain rescaled FP16 tensors; later layer variants must account for the full layer state.

Sources: [Qwen AWQ checkpoint](https://huggingface.co/Qwen/Qwen2.5-1.5B-Instruct-AWQ), [AutoAWQ compatibility](https://github.com/casper-hansen/AutoAWQ), [AutoAWQ GEMM implementation](https://github.com/casper-hansen/AutoAWQ/blob/v0.2.9/awq/modules/linear/gemm.py), [PyTorch installation](https://pytorch.org/get-started/previous-versions/).